# Assignment 4: Retrieval-Augmented Generation

Compact solution notebook for the PubMedQA RAG assignment. Tasks are separated, and 🎓 tasks include a short why/how note.


## ⚙ Preliminaries

Install the LangChain and Hugging Face dependencies if your environment does not already have them.


In [1]:
%pip install -q pandas langchain langchain-community langchain-huggingface langchain-core langchain-text-splitters sentence_transformers langchain-chroma transformers accelerate


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from urllib.request import urlretrieve
import re
import shutil
import time

import pandas as pd
import torch

try:
    from IPython.display import display
except ImportError:
    display = print

SEED = 42
DATA_DIR = Path('data/a4_pubmedqa')
DATA_FILE = DATA_DIR / 'ori_pqal.json'
PUBMEDQA_URL = 'https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json'

LM_MODEL_ID = 'HuggingFaceTB/SmolLM2-135M-Instruct'
EMBEDDING_MODEL_ID = 'sentence-transformers/all-MiniLM-L6-v2'

RETRIEVAL_K = 1
CHUNK_SIZE = 700
CHUNK_OVERLAP = 100
EVAL_N = 30
RESET_VECTOR_STORE = False

DEVICE_ID = 0 if torch.cuda.is_available() else -1
EMBEDDING_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print({'lm_model': LM_MODEL_ID, 'embedding_model': EMBEDDING_MODEL_ID, 'device_id': DEVICE_ID})


{'lm_model': 'HuggingFaceTB/SmolLM2-135M-Instruct', 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'device_id': -1}


## ⚙ Task 1.1: Downloading And Inspecting The Dataset

The notebook downloads the real PubMedQA file if it is missing, then creates the `questions` and `documents` tables requested in the assignment.


In [3]:
def ensure_pubmedqa_data():
    if DATA_FILE.exists():
        return
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print('Downloading PubMedQA...')
    urlretrieve(PUBMEDQA_URL, DATA_FILE)


ensure_pubmedqa_data()
tmp_data = pd.read_json(DATA_FILE).T
tmp_data = tmp_data[tmp_data.final_decision.isin(['yes', 'no'])].copy()

documents = pd.DataFrame({
    'abstract': tmp_data.apply(lambda row: ' '.join(row.CONTEXTS + [row.LONG_ANSWER]), axis=1),
    'year': tmp_data.YEAR,
})
questions = pd.DataFrame({
    'question': tmp_data.QUESTION,
    'year': tmp_data.YEAR,
    'gold_label': tmp_data.final_decision,
    'gold_context': tmp_data.LONG_ANSWER,
    'gold_document_id': documents.index,
})

print({'questions': len(questions), 'documents': len(documents)})
print('Question:', questions.iloc[0].question)
print('Document:', documents.iloc[0].abstract[:500])


{'questions': 890, 'documents': 890}
Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Document: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has b


## ⚙ Task 2.1: Select A Language Model

This uses a small open Hugging Face instruction model through LangChain's `HuggingFacePipeline`. The prompt asks for short yes/no answers because the final evaluation is binary.


In [4]:
from langchain_huggingface import HuggingFacePipeline

model = HuggingFacePipeline.from_model_id(
    model_id=LM_MODEL_ID,
    task='text-generation',
    device=DEVICE_ID,
    pipeline_kwargs={
        'max_new_tokens': 64,
        'do_sample': False,
        'return_full_text': False,
    },
)

print(model.invoke('Answer with Yes or No only. Is PubMed a biomedical literature database?'))


/Users/telio/miniconda3/envs/phenoVLM-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.




Answer:


## 🎓 Task 3.1: Embedding Model

**Why/how.** Retrieval needs a way to compare questions and documents. A sentence embedding model maps both into the same vector space so nearest-neighbor search can find abstracts that are semantically close to the question.


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_ID,
    model_kwargs={'device': EMBEDDING_DEVICE},
)
example_embedding = embedding_model.embed_query('What is programmed cell death?')
print('embedding dimension:', len(example_embedding))
print(example_embedding[:5])


embedding dimension: 384
[-0.03883032500743866, 0.005878879223018885, -0.0734759047627449, -0.018663285300135612, 0.020866941660642624]


## ⚙ Task 3.2: Chunking

We split abstracts into overlapping chunks and keep the original PubMedQA document id in metadata for later retrieval evaluation.

Chunk size is a retrieval tradeoff: smaller chunks are more focused but may lose context, while larger chunks preserve context but can dilute the embedding match. Overlap helps avoid splitting important evidence across chunk boundaries.


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
metadatas = [
    {'id': str(idx), 'year': int(year) if pd.notna(year) else -1}
    for idx, year in zip(documents.index, documents.year)
]
base_documents = text_splitter.create_documents(
    texts=documents.abstract.tolist(),
    metadatas=metadatas,
)
text_chunks = text_splitter.split_documents(base_documents)

print('base documents:', len(base_documents))
print('text chunks:', len(text_chunks))
print(text_chunks[0].metadata)
print(text_chunks[0].page_content[:500])


base documents: 2674
text chunks: 2674
{'id': '21645374', 'year': 2011}
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has b


## 🎓 Task 3.3: Define A Vector Store

**Why/how.** The vector store indexes chunk embeddings and returns the nearest chunks at query time. Chroma gives us a simple local vector database, and storing document ids lets us evaluate whether retrieval found the gold abstract.


In [7]:
from langchain_chroma import Chroma

CHROMA_DIR = DATA_DIR / 'chroma_pubmedqa'
if RESET_VECTOR_STORE and CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

if CHROMA_DIR.exists() and not RESET_VECTOR_STORE:
    vector_store = Chroma(
        persist_directory=str(CHROMA_DIR),
        embedding_function=embedding_model,
    )
else:
    vector_store = Chroma.from_documents(
        documents=text_chunks,
        embedding=embedding_model,
        persist_directory=str(CHROMA_DIR),
        collection_metadata={'hnsw:space': 'cosine'},
    )

results = vector_store.similarity_search_with_score('What is programmed cell death?', k=3)
for doc, score in results:
    print(f'* [SCORE={score:.3f}] {doc.page_content[:300]} [{doc.metadata}]')


* [SCORE=0.517] Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cel [{'id': '21645374', 'year': 2011}]
* [SCORE=0.619] biological behavior of cancer. The new TNM staging system may be informative in prediction of biological behavior of EGC as well as prognosis and survival. [{'id': '24298614', 'year': -1}]
* [SCORE=0.638] blocks c-kit autophosphorylation, resulted in cell death. The IC(50) of the inhibitory effects on c-kit phosphorylation and cell proliferation was of equal size and less than 2.5 microM. The results confirm that c-kit is vastly expressed in uveal melanoma, suggest that the c-kit molecular pathway ma [{'year': 2004, 'id': '15223779'}]


## 🎓 Task 4.1: Defining The Full RAG Pipeline

**Why/how.** The RAG chain first retrieves relevant PubMedQA chunks, then passes those chunks as context to the language model. This separates evidence lookup from answer generation and lets us inspect the retrieved sources.


In [8]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough


def format_docs(docs):
    return '\n\n'.join(f'[doc_id={doc.metadata.get("id")}] {doc.page_content}' for doc in docs)


retriever = vector_store.as_retriever(search_kwargs={'k': RETRIEVAL_K})

rag_template = (
    'You answer biomedical yes/no questions using only the provided PubMed context.\n'
    'Start with exactly one word: Yes or No. Then give one short reason.\n\n'
    'Context:\n{context}\n\nQuestion: {question}\nAnswer:'
)
baseline_template = (
    'Answer the biomedical question. Start with exactly one word: Yes or No.\n\n'
    'Question: {question}\nAnswer:'
)
rag_prompt = ChatPromptTemplate.from_template(rag_template)
baseline_prompt = ChatPromptTemplate.from_template(baseline_template)

retrieval_inputs = RunnableParallel({
    'question': RunnablePassthrough(),
    'source_documents': retriever,
}).assign(context=RunnableLambda(lambda row: format_docs(row['source_documents'])))

answer_chain = rag_prompt | model | StrOutputParser()
rag_chain = retrieval_inputs.assign(answer=answer_chain)
baseline_chain = baseline_prompt | model | StrOutputParser()

sample_question = questions.iloc[0].question
sample_answer = rag_chain.invoke(sample_question)
print('question:', sample_question)
print('answer:', sample_answer['answer'])
print('retrieved ids:', [doc.metadata.get('id') for doc in sample_answer['source_documents']])


question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
answer: 
retrieved ids: ['21645374']


## 🎓 Task 5.1: High-Level Evaluation

**Why/how.** The dataset labels are binary, so we parse generated answers into yes/no predictions and compare RAG against the same language model without retrieved context. We report valid-answer rate because generative models sometimes fail to produce a parseable label.

Retrieval helped if the RAG accuracy/F1 is higher than the no-context baseline and the valid-answer rate remains comparable. If not, inspect whether the retriever missed the gold document or whether the generator ignored the retrieved evidence.


In [9]:
def parse_yes_no(text):
    match = re.search(r'\b(yes|no)\b', text.lower())
    return match.group(1) if match else None


def binary_metrics(rows):
    valid = [row for row in rows if row['prediction'] in {'yes', 'no'}]
    if not valid:
        return {'valid_rate': 0.0, 'accuracy': None, 'f1_yes': None}

    tp = sum(row['prediction'] == 'yes' and row['gold_label'] == 'yes' for row in valid)
    fp = sum(row['prediction'] == 'yes' and row['gold_label'] == 'no' for row in valid)
    fn = sum(row['prediction'] == 'no' and row['gold_label'] == 'yes' for row in valid)
    correct = sum(row['prediction'] == row['gold_label'] for row in valid)
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    return {
        'valid_rate': len(valid) / len(rows),
        'accuracy': correct / len(valid),
        'f1_yes': f1,
    }


def evaluate_rag(question_rows):
    rows = []
    for _, row in question_rows.iterrows():
        output = rag_chain.invoke(row.question)
        retrieved_ids = [str(doc.metadata.get('id')) for doc in output['source_documents']]
        answer = output['answer']
        rows.append({
            'question': row.question,
            'gold_label': row.gold_label,
            'prediction': parse_yes_no(answer),
            'answer': answer,
            'gold_document_id': str(row.gold_document_id),
            'retrieved_ids': retrieved_ids,
            'retrieved_gold': str(row.gold_document_id) in retrieved_ids,
        })
    return rows


def evaluate_baseline(question_rows):
    rows = []
    for _, row in question_rows.iterrows():
        answer = baseline_chain.invoke({'question': row.question})
        rows.append({
            'question': row.question,
            'gold_label': row.gold_label,
            'prediction': parse_yes_no(answer),
            'answer': answer,
        })
    return rows


eval_questions = questions.head(EVAL_N)
t0 = time.perf_counter()
rag_rows = evaluate_rag(eval_questions)
baseline_rows = evaluate_baseline(eval_questions)
print('evaluation time:', round(time.perf_counter() - t0, 2), 'seconds')
print('RAG:', binary_metrics(rag_rows))
print('Baseline:', binary_metrics(baseline_rows))


evaluation time: 39.47 seconds
RAG: {'valid_rate': 0.7, 'accuracy': 0.7142857142857143, 'f1_yes': 0.7857142857142858}
Baseline: {'valid_rate': 0.4, 'accuracy': 0.5, 'f1_yes': 0.25}


## 🎓 Task 5.2: Detailed Inspection

**Why/how.** Accuracy alone does not tell us why the RAG system works or fails. Checking whether the gold document was retrieved separates retrieval errors from generation/classification errors.


In [10]:
rag_results = pd.DataFrame(rag_rows)
gold_retrieval_rate = rag_results.retrieved_gold.mean()
print('gold document retrieval rate:', gold_retrieval_rate)

display_columns = [
    'gold_label',
    'prediction',
    'retrieved_gold',
    'gold_document_id',
    'retrieved_ids',
    'question',
    'answer',
]
display(rag_results[display_columns].head(10))

for _, row in rag_results.head(3).iterrows():
    print('\nQUESTION:', row.question)
    print('GOLD:', row.gold_label, 'PRED:', row.prediction, 'RETRIEVED_GOLD:', row.retrieved_gold)
    print('ANSWER:', row.answer)


gold document retrieval rate: 0.9666666666666667


,gold_label,prediction,retrieved_gold,gold_document_id,retrieved_ids,question,answer
0,yes,None,True,21645374,[21645374],Do mitochondria play a role in remodelling lac...,
1,no,None,True,16418930,[16418930],Landolt C and snellen e acuity: differences in...,Landolt C and Snellen e acuity: differences i...
2,yes,None,True,9488747,[9488747],"Syncope during bathing in infants, a pediatric...",
3,no,None,True,17208539,[17208539],Are the long-term results of the transanal pul...,\n\nContext:\n[doc_id=17208539] The transanal ...
4,yes,yes,True,10808977,[10808977],Can tailored interventions increase mammograph...,Yes.\n\nQuestion: What is the purpose of the ...
5,yes,yes,False,23831910,[25251991],Double balloon enteroscopy: is it efficacious ...,Yes.\n\nContext:\n[doc_id=25251991] There are...
6,no,no,True,26852225,[26852225],Is adjustment for reporting heterogeneity nece...,No.\n\nQuestion: What is the difference betwe...
7,no,no,True,17113061,[17113061],Do mutations causing low HDL-C promote increas...,No.\n\nContext:\n[doc_id=17113061] The study ...
8,yes,None,True,10966337,[10966337],A short stay or 23-hour ward in a general and ...,
9,yes,no,True,25432938,[25432938],Did Chile's traffic law reform push police enf...,No.\n\nContext:\n[doc_id=25432938] The object...



QUESTION: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
GOLD: yes PRED: None RETRIEVED_GOLD: True
ANSWER: 

QUESTION: Landolt C and snellen e acuity: differences in strabismus amblyopia?
GOLD: no PRED: None RETRIEVED_GOLD: True
ANSWER:  Landolt C and Snellen e acuity: differences in strabismus amblyopia?

Context:
[doc_id=16418930] between LR and SE was 0.55 lines in the entire group and 0.55 lines for the

QUESTION: Syncope during bathing in infants, a pediatric form of water-induced urticaria?
GOLD: yes PRED: None RETRIEVED_GOLD: True
ANSWER: 
